In [ ]:
from openai import OpenAI
import time
import json
import pandas as pd
import re

# Initialize OpenAI client
openai_client = OpenAI(api_key="your-openai-api-key-here")

# Load JSON file
with open('/content/test_collection_IPC_147.json', 'r') as f:
    data = json.load(f)

# Process first 5 documents
case_data = []
for filename, case_details in list(data.items())[:5]:
    case_data.append({
        "doc_id": filename,
        "fact": case_details.get("fact", "")
    })

test_df = pd.DataFrame(case_data)

def model_openai(model_name, num_tokens, sp, up):
    while True:
        try:
            completion = openai_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": sp}, {"role": "user", "content": up}],
                temperature=0,
                max_tokens=num_tokens,
                top_p=0.0,
                stream=False,
                stop=None,
            )
            break
        except Exception as e:
            print(f"Error: {e}. Retrying in 2 minutes...")
            time.sleep(120)
    return completion.choices[0].message.content

model_name = "gpt-4o-mini"  # Changed to OpenAI model
num_tokens = 6000

prompt = """You are an expert in Indian Penal Code (IPC) statutes. Analyze the legal case fact pattern and STRICTLY follow these rules:

1. ONLY consider these 7 IPC sections:
"Indian Penal Code 498A": " Whoever, being the husband or the relative of the husband of a woman, subjects such woman to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine.",
"Indian Penal Code 506": " Whoever commits the offence of criminal intimidation shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both",
"Indian Penal Code 147": " Whoever is guilty of rioting, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.",
"Indian Penal Code 201": " Whoever, knowing or having reason to believe that an offence has been committed, causes any evidence of the commission of that offence to disappear, with the intention of screening the offender from legal punishment, or with that intention gives any information respecting the offence which he knows or believes to be false;",
"Indian Penal Code 302": " Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 376": " Whoever, except in the cases provided for in sub-section (2), commits rape, shall be punished with rigorous imprisonment of either description for a term which shall not be less than ten years, but which may extend to imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 420": " Whoever cheats and thereby dishonestly induces the person deceived to deliver any property to any person, or to make, alter or destroy the whole or any part of a valuable security, or anything which is signed or sealed, and which is capable of being converted into a valuable security, shall be punished with imprisonment of either description for a term which may extend to seven years, and shall also be liable to fine."

2. For EACH applicable section:
- Extract EXACT quoted phrases from facts that directly match the statute's requirements
- DO NOT CONSIDER ANY OTHER IPC SECTIONS UNDER ANY CIRCUMSTANCES

3. Format response as JSON list with:
- "statute": Full IPC section name from above list
- "legal_reasoning": EXACT quoted text from facts

Example Response:
json
[{
    "statute": "Indian Penal Code 302",
    "legal_reasoning": "victim was stabbed three times in the chest"
}]
"""

ALLOWED_SECTIONS = {"498A", "506", "147", "201", "302", "376", "420"}

def format_response(response):
    # Extract JSON from markdown code block
    json_match = re.search(r'json\n(.*?)\n', response, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group(1))
        except json.JSONDecodeError:
            parsed = []
    else:
        # Fallback to direct parsing
        try:
            parsed = json.loads(response)
        except json.JSONDecodeError:
            parsed = []

    # Regex validation for strict pattern matching
    pattern = r'''
    \{\s*
        "statute":\s*"Indian Penal Code (498A|506|147|201|302|376|420)"\s*,
        \s*"legal_reasoning":\s*"(.*?)"
    \s*\}'''
    matches = re.findall(pattern, response, re.DOTALL | re.VERBOSE)

    regex_parsed = []
    for match in matches:
        section_num = match[0]
        reasoning = match[1].strip()
        regex_parsed.append({
            "statute": f"Indian Penal Code {section_num}",
            "legal_reasoning": reasoning
        })

    # Combine both parsing methods
    combined = parsed + regex_parsed

    # Validate and filter results
    validated = []
    seen = set()
    for item in combined:
        try:
            section_num = item["statute"].split()[-1]
            if section_num in ALLOWED_SECTIONS:
                if section_num not in seen:
                    seen.add(section_num)
                    validated.append({
                        "statute": item["statute"],
                        "legal_reasoning": item["legal_reasoning"]
                    })
        except (KeyError, AttributeError):
            continue

    return validated

def main():
    results = {}

    for i in range(min(5, len(test_df))):
        doc_id = test_df.iloc[i]['doc_id']
        fact_text = test_df.iloc[i]['fact']

        response = model_openai(model_name, num_tokens,
                              prompt,
                              f"Case Facts: {fact_text}")

        try:
            analysis = format_response(response)
            results[doc_id] = {
                "analysis": analysis,
                "full_response": response
            }
        except Exception as e:
            print(f"Error processing {doc_id}: {str(e)}")
            results[doc_id] = {
                "error": str(e),
                "full_response": response
            }

        print(f"\nProcessed {doc_id}")
        print("Analysis Results:")
        for entry in results[doc_id].get('analysis', []):
            print(f"Statute: {entry['statute']}")
            print(f"Reasoning: {entry['legal_reasoning']}")
        print("-" * 50)

    # Save results
    with open("strict_ipc_analysis.json", "w") as f:
        json.dump(results, f, indent=4)
    print("\nFinal results saved to strict_ipc_analysis.json")

if _name_ == "_main_":
    main()